In [2]:
import pandas as pd
import re

results = pd.read_csv("data/raw/results.csv")
shootouts = pd.read_csv("data/raw/shootouts.csv")
elo = pd.read_csv("data/raw/elo_ratings.csv")
wc2026 = pd.read_csv("data/raw/wc2026_matches.csv")
importance = pd.read_csv("data/raw/fifa_match_importance_weights.csv")

for name, df in [("results", results), ("shootouts", shootouts), ("elo", elo), ("wc2026", wc2026)]:
    print(name, df.shape)

results (49499, 9)
shootouts (680, 5)
elo (244, 3)
wc2026 (85, 3)


In [3]:
def parse_score(score_str):
    score_str = str(score_str).strip()
    went_to_et = "a.e.t." in score_str
    # remove any parenthetical notes like "(a.e.t.)"
    clean = re.sub(r"\(.*?\)", "", score_str).strip()
    # split on en-dash or regular hyphen
    parts = re.split(r"[–-]", clean)
    home_score = int(parts[0].strip())
    away_score = int(parts[1].strip())
    return pd.Series([home_score, away_score, went_to_et])

wc2026[["home_score", "away_score", "extra_time"]] = wc2026["score"].apply(parse_score)
wc2026.head(20)

,home_team,score,away_team,home_score,away_score,extra_time
0,Mexico,2–0,South Africa,2,0,False
1,South Korea,2–1,Czech Republic,2,1,False
2,Czech Republic,1–1,South Africa,1,1,False
3,Mexico,1–0,South Korea,1,0,False
4,Czech Republic,0–3,Mexico,0,3,False
5,South Africa,1–0,South Korea,1,0,False
6,Canada,1–1,Bosnia and Herzegovina,1,1,False
7,Qatar,1–1,Switzerland,1,1,False
8,Switzerland,4–1,Bosnia and Herzegovina,4,1,False
9,Canada,6–0,Qatar,6,0,False


In [4]:
wc2026[wc2026["extra_time"] == True]

,home_team,score,away_team,home_score,away_score,extra_time
74,Germany,1–1 (a.e.t.),Paraguay,1,1,True
75,Netherlands,1–1 (a.e.t.),Morocco,1,1,True
80,Belgium,3–2 (a.e.t.),Senegal,3,2,True


In [5]:
results_teams = set(results["home_team"]).union(set(results["away_team"]))
wc2026_teams = set(wc2026["home_team"]).union(set(wc2026["away_team"]))
elo_teams = set(elo["team"])

print("Teams in wc2026 but NOT in results:", wc2026_teams - results_teams)
print()
print("Teams in wc2026 but NOT in elo:", wc2026_teams - elo_teams)

Teams in wc2026 but NOT in results: set()

Teams in wc2026 but NOT in elo: {'Czech Republic'}


In [6]:
[t for t in elo_teams if "czech" in t.lower() or "czechia" in t.lower()]

['Czechia']

In [7]:
elo["team"] = elo["team"].replace({"Czechia": "Czech Republic"})  # adjust if the printed name differs

# re-verify
elo_teams = set(elo["team"])
print("Teams in wc2026 but NOT in elo:", wc2026_teams - elo_teams)

Teams in wc2026 but NOT in elo: set()


In [8]:
print(results.dtypes)
results["date"] = pd.to_datetime(results["date"])
print(results["date"].min(), "to", results["date"].max())
results.head()

date           object
home_team      object
away_team      object
home_score    float64
away_score    float64
tournament     object
city           object
country        object
neutral          bool
dtype: object
1872-11-30 00:00:00 to 2026-07-06 00:00:00


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [9]:
# Historical results already have the fields we need
results_clean = results[["date", "home_team", "away_team", "home_score", "away_score", "tournament", "neutral"]].copy()

# The 2026 data needs a date and tournament label added (Wikipedia table didn't give us dates directly)
wc2026_clean = wc2026[["home_team", "away_team", "home_score", "away_score", "extra_time"]].copy()
wc2026_clean["tournament"] = "FIFA World Cup"
wc2026_clean["neutral"] = True  # co-hosted, but not each team's home country — refine later if needed

print(results_clean.shape, wc2026_clean.shape)
results_clean.tail()

(49499, 7) (85, 7)


,date,home_team,away_team,home_score,away_score,tournament,neutral
49494,2026-07-04,Paraguay,France,NaN,NaN,FIFA World Cup,True
49495,2026-07-05,Brazil,Norway,NaN,NaN,FIFA World Cup,True
49496,2026-07-05,Mexico,England,NaN,NaN,FIFA World Cup,False
49497,2026-07-06,Portugal,Spain,NaN,NaN,FIFA World Cup,True
49498,2026-07-06,United States,Belgium,NaN,NaN,FIFA World Cup,False


In [10]:
wc2026_in_results = results[
    (results["tournament"] == "FIFA World Cup") &
    (results["date"] >= "2026-06-01")
]
print(wc2026_in_results.shape)
wc2026_in_results.head(10)

(94, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49405,2026-06-11,Mexico,South Africa,2.0,0.0,FIFA World Cup,Mexico City,Mexico,False
49406,2026-06-11,South Korea,Czech Republic,2.0,1.0,FIFA World Cup,Zapopan,Mexico,True
49407,2026-06-12,Canada,Bosnia and Herzegovina,1.0,1.0,FIFA World Cup,Toronto,Canada,False
49408,2026-06-12,United States,Paraguay,4.0,1.0,FIFA World Cup,Inglewood,United States,False
49409,2026-06-13,Qatar,Switzerland,1.0,1.0,FIFA World Cup,Santa Clara,United States,True
49410,2026-06-13,Brazil,Morocco,1.0,1.0,FIFA World Cup,East Rutherford,United States,True
49411,2026-06-13,Haiti,Scotland,0.0,1.0,FIFA World Cup,Foxborough,United States,True
49412,2026-06-13,Australia,Turkey,2.0,0.0,FIFA World Cup,Vancouver,Canada,True
49413,2026-06-14,Germany,Curaçao,7.0,1.0,FIFA World Cup,Houston,United States,True
49414,2026-06-14,Ivory Coast,Ecuador,1.0,0.0,FIFA World Cup,Philadelphia,United States,True


In [11]:
matches = results.copy()
matches = matches.dropna(subset=["home_score", "away_score"])  # drop unplayed/future fixtures for now
matches["home_score"] = matches["home_score"].astype(int)
matches["away_score"] = matches["away_score"].astype(int)
print(matches.shape)
matches.tail()

(49490, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49485,2026-07-01,Belgium,Senegal,3,2,FIFA World Cup,Seattle,United States,True
49486,2026-07-01,United States,Bosnia and Herzegovina,2,0,FIFA World Cup,Santa Clara,United States,False
49487,2026-07-02,Spain,Austria,3,0,FIFA World Cup,Inglewood,United States,True
49488,2026-07-02,Portugal,Croatia,2,1,FIFA World Cup,Toronto,Canada,True
49489,2026-07-02,Switzerland,Algeria,2,0,FIFA World Cup,Vancouver,Canada,True


In [13]:
# rebuild et_lookup with date info from your original wc2026 scrape
# first, get the 2026 dates from `matches` itself, matched by tournament + team pair
et_lookup_2026 = matches[
    (matches["tournament"] == "FIFA World Cup") &
    (matches["date"] >= "2026-06-01") &
    (matches["home_team"].isin(["Germany", "Netherlands", "Belgium"])) &
    (matches["away_team"].isin(["Paraguay", "Morocco", "Senegal"]))
][["date", "home_team", "away_team"]].copy()

et_lookup_2026["is_et"] = True
print(et_lookup_2026)

# reset extra_time column and redo the merge properly, keyed on date + teams
matches = matches.drop(columns=["extra_time"])  # remove the bad column
matches = matches.merge(et_lookup_2026, on=["date", "home_team", "away_team"], how="left")
matches["extra_time"] = matches["is_et"].fillna(False)
matches = matches.drop(columns=["is_et"])

print(matches["extra_time"].sum(), "matches correctly flagged as extra time")
matches[matches["extra_time"] == True]

            date    home_team away_team  is_et
49479 2026-06-29      Germany  Paraguay   True
49480 2026-06-29  Netherlands   Morocco   True
49485 2026-07-01      Belgium   Senegal   True
3 matches correctly flagged as extra time


C:\Users\tishy\AppData\Local\Temp\ipykernel_55220\4211479400.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  matches["extra_time"] = matches["is_et"].fillna(False)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,extra_time
49479,2026-06-29,Germany,Paraguay,1,1,FIFA World Cup,Foxborough,United States,True,True
49480,2026-06-29,Netherlands,Morocco,1,1,FIFA World Cup,Guadalupe,Mexico,True,True
49485,2026-07-01,Belgium,Senegal,3,2,FIFA World Cup,Seattle,United States,True,True


In [14]:
matches.to_csv("data/processed/matches.csv", index=False)
print(matches.shape)
matches.dtypes

(49490, 10)


date          datetime64[ns]
home_team             object
away_team             object
home_score             int32
away_score             int32
tournament            object
city                  object
country               object
neutral                 bool
extra_time              bool
dtype: object

In [15]:
matches = matches.merge(
    elo[["team", "elo_rating"]].rename(columns={"team": "home_team", "elo_rating": "home_elo"}),
    on="home_team", how="left"
)
matches = matches.merge(
    elo[["team", "elo_rating"]].rename(columns={"team": "away_team", "elo_rating": "away_elo"}),
    on="away_team", how="left"
)

print(matches[["home_team", "away_team", "home_elo", "away_elo"]].isna().sum())
matches.tail()

home_team       0
away_team       0
home_elo     2249
away_elo     2484
dtype: int64


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,extra_time,home_elo,away_elo
49485,2026-07-01,Belgium,Senegal,3,2,FIFA World Cup,Seattle,United States,True,True,1910.0,1816.0
49486,2026-07-01,United States,Bosnia and Herzegovina,2,0,FIFA World Cup,Santa Clara,United States,False,False,1798.0,1605.0
49487,2026-07-02,Spain,Austria,3,0,FIFA World Cup,Inglewood,United States,True,False,2159.0,1821.0
49488,2026-07-02,Portugal,Croatia,2,1,FIFA World Cup,Toronto,Canada,True,False,2013.0,1882.0
49489,2026-07-02,Switzerland,Algeria,2,0,FIFA World Cup,Vancouver,Canada,True,False,1943.0,1756.0


In [16]:
shootouts_flag = shootouts[["date", "home_team", "away_team", "winner"]].copy()
shootouts_flag["date"] = pd.to_datetime(shootouts_flag["date"])
shootouts_flag = shootouts_flag.rename(columns={"winner": "shootout_winner"})

matches = matches.merge(shootouts_flag, on=["date", "home_team", "away_team"], how="left")
matches["went_to_shootout"] = matches["shootout_winner"].notna()

print(matches["went_to_shootout"].sum(), "matches decided by penalties")
matches[matches["went_to_shootout"] == True].tail()

679 matches decided by penalties


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,extra_time,home_elo,away_elo,shootout_winner,went_to_shootout
49252,2026-03-31,Bosnia and Herzegovina,Italy,1,1,FIFA World Cup qualification,Zenica,Bosnia and Herzegovina,False,False,1605.0,1869.0,Bosnia and Herzegovina,True
49255,2026-03-31,Czech Republic,Denmark,2,2,FIFA World Cup qualification,Prague,Czech Republic,False,False,1680.0,1869.0,Czech Republic,True
49364,2026-06-06,Lithuania,Latvia,1,1,Baltic Cup,Kaunas,Lithuania,False,False,1279.0,1288.0,Lithuania,True
49479,2026-06-29,Germany,Paraguay,1,1,FIFA World Cup,Foxborough,United States,True,True,1908.0,1823.0,Paraguay,True
49480,2026-06-29,Netherlands,Morocco,1,1,FIFA World Cup,Guadalupe,Mexico,True,True,1971.0,1886.0,Morocco,True


In [17]:
future_fixtures = results[results["home_score"].isna()].copy()
future_fixtures = future_fixtures[future_fixtures["tournament"] == "FIFA World Cup"]
future_fixtures.to_csv("data/processed/future_fixtures.csv", index=False)
print(future_fixtures.shape)
future_fixtures

(9, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49490,2026-07-03,Australia,Egypt,NaN,NaN,FIFA World Cup,Arlington,United States,True
49491,2026-07-03,Argentina,Cape Verde,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True
49492,2026-07-03,Colombia,Ghana,NaN,NaN,FIFA World Cup,Kansas City,United States,True
49493,2026-07-04,Canada,Morocco,NaN,NaN,FIFA World Cup,Houston,United States,True
49494,2026-07-04,Paraguay,France,NaN,NaN,FIFA World Cup,Philadelphia,United States,True
49495,2026-07-05,Brazil,Norway,NaN,NaN,FIFA World Cup,East Rutherford,United States,True
49496,2026-07-05,Mexico,England,NaN,NaN,FIFA World Cup,Mexico City,Mexico,False
49497,2026-07-06,Portugal,Spain,NaN,NaN,FIFA World Cup,Dallas,United States,True
49498,2026-07-06,United States,Belgium,NaN,NaN,FIFA World Cup,Seattle,United States,False


In [18]:
matches.to_csv("data/processed/matches_full.csv", index=False)
print(matches.shape)
matches.columns.tolist()

(49490, 14)


['date',
 'home_team',
 'away_team',
 'home_score',
 'away_score',
 'tournament',
 'city',
 'country',
 'neutral',
 'extra_time',
 'home_elo',
 'away_elo',
 'shootout_winner',
 'went_to_shootout']